In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier

from sklearn.metrics import accuracy_score
from scipy.stats import ttest_rel

import ipywidgets as widgets
from IPython.display import display


In [ ]:
models = {
    "Logistic": make_pipeline(
        StandardScaler(),
        LogisticRegression(C=1.0, solver="liblinear")
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        random_state=0,
        n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=0
    ),
    "BaggingTree": BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=50,
        random_state=0,
        n_jobs=-1
    )
}


In [ ]:
def evaluate(class_sep=1.0, flip_y=0.0, n_samples=1000, n_runs=10):
    scores = {k: [] for k in models}

    for seed in range(n_runs):
        X, y = make_classification(
            n_samples=n_samples,
            n_features=20,
            n_informative=5,
            n_redundant=2,
            class_sep=class_sep,
            flip_y=flip_y,
            random_state=seed
        )

        Xtr, Xte, ytr, yte = train_test_split(
            X, y, test_size=0.3, stratify=y, random_state=seed
        )

        for name, model in models.items():
            model.fit(Xtr, ytr)
            pred = model.predict(Xte)
            scores[name].append(accuracy_score(yte, pred))

    return {k: np.array(v) for k, v in scores.items()}


In [ ]:
def plot_results(class_sep, flip_y, n_samples):
    results = evaluate(class_sep, flip_y, n_samples)

    means = {k: v.mean() for k, v in results.items()}
    stds  = {k: v.std()  for k, v in results.items()}

    df = pd.DataFrame({
        "mean": means,
        "std": stds
    })

    display(df)

    # Test estadístico
    for ens in ["RandomForest", "GradientBoosting", "BaggingTree"]:
        t, p = ttest_rel(results["Logistic"], results[ens])
        if p < 0.05 and results["Logistic"].mean() > results[ens].mean():
            print(f"✔ Logistic supera significativamente a {ens} (p={p:.4f})")

    # Plot
    plt.figure(figsize=(8,5))
    plt.bar(df.index, df["mean"], yerr=df["std"], capsize=5)
    plt.ylabel("Accuracy")
    plt.title("Modelo simple vs Ensembles")
    plt.grid(axis="y")
    plt.show()


In [ ]:
ui = widgets.VBox([
    widgets.FloatSlider(
        value=1.0, min=0.2, max=2.5, step=0.1,
        description="class_sep"
    ),
    widgets.FloatSlider(
        value=0.0, min=0.0, max=0.5, step=0.05,
        description="flip_y"
    ),
    widgets.IntSlider(
        value=1000, min=50, max=5000, step=50,
        description="n_samples"
    )
])

out = widgets.interactive_output(
    plot_results,
    {
        "class_sep": ui.children[0],
        "flip_y": ui.children[1],
        "n_samples": ui.children[2]
    }
)

display(ui, out)


Output()